# 04_export_report.ipynb - Export Report & Dashboard

Notebook ini untuk:
- Generate executive summary report
- Export data ke Excel dengan multiple sheets
- Buat dashboard statis untuk manajemen
- Archive hasil analisis bulanan

**ATURAN:** Notebook ini idempotent - bisa dijalankan ulang dari awal.

In [ ]:
# ── SETUP ────────────────────────────────────────────────────────────────
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))

import pandas as pd
from datetime import datetime
from config import DATA_DIR, OUT_DIR, REF_DIR, GHOST_DAYS, CHURN_THRESHOLD

print("✅ Setup complete")

In [ ]:
# ── LOAD ALL ANALYSIS RESULTS ───────────────────────────────────────────
print("📂 Loading analysis results...")

df_ghost = pd.read_csv(OUT_DIR / "ghost_outlets.csv")
df_route = pd.read_csv(OUT_DIR / "route_compliance.csv")
df_sales = pd.read_csv(OUT_DIR / "sales_performance.csv")
df_prospect = pd.read_csv(OUT_DIR / "prospect_score.csv")

# Load raw data untuk summary
df_doccall = pd.read_parquet(DATA_DIR / "doccall.parquet")
df_customer = pd.read_parquet(DATA_DIR / "customer.parquet")

print(f"  ✅ ghost_outlets: {len(df_ghost):,} rows")
print(f"  ✅ route_compliance: {len(df_route):,} rows")
print(f"  ✅ sales_performance: {len(df_sales):,} rows")
print(f"  ✅ prospect_score: {len(df_prospect):,} rows")

In [ ]:
# ── GENERATE EXECUTIVE SUMMARY ──────────────────────────────────────────
print("\n" + "="*60)
print("📊 EXECUTIVE SUMMARY")
print("="*60)

# Hitung KPI utama
total_outlets = len(df_customer)
ghost_count = len(df_ghost)
ghost_rate = ghost_count / total_outlets * 100 if total_outlets > 0 else 0

churn_count = len(df_sales[df_sales['churn_risk'] == True])
churn_rate = churn_count / len(df_sales) * 100 if len(df_sales) > 0 else 0

# Visit statistics
visit_counts = df_doccall.groupby('szCustomerId').size()
avg_visits = visit_counts.mean()
outlets_with_visits = len(visit_counts)

# Prospect scoring
high_potential = len(df_prospect[df_prospect['prospect_score'] >= 70])
medium_potential = len(df_prospect[(df_prospect['prospect_score'] >= 40) & (df_prospect['prospect_score'] < 70)])
low_potential = len(df_prospect[df_prospect['prospect_score'] < 40])

print(f"\n🏪 Total Outlet: {total_outlets:,}")
print(f"\n👻 Ghost Outlets (> {GHOST_DAYS} hari):")
print(f"   Count: {ghost_count:,} ({ghost_rate:.1f}%)")
print(f"\n⚠️  Churn Risk (< {CHURN_THRESHOLD*100:.0f}%):")
print(f"   Count: {churn_count:,} ({churn_rate:.1f}%)")
print(f"\n🛣️  Kunjungan Sales:")
print(f"   Outlet dikunjungi: {outlets_with_visits:,}")
print(f"   Rata-rata kunjungan: {avg_visits:.1f}x")
print(f"\n🎯 Prospect Scoring:")
print(f"   High (≥70): {high_potential:,}")
print(f"   Medium (40-69): {medium_potential:,}")
print(f"   Low (<40): {low_potential:,}")

In [ ]:
# ── BREAKDOWN PER BRANCH ────────────────────────────────────────────────
print("\n" + "="*60)
print("📍 BREAKDOWN PER BRANCH")
print("="*60)

# Group by branch
branch_summary = df_customer.groupby('szBranchId').size().reset_index(name='total_outlets')

# Merge dengan ghost count per branch
ghost_by_branch = df_ghost.groupby('szBranchId').size().reset_index(name='ghost_count')
branch_summary = branch_summary.merge(ghost_by_branch, on='szBranchId', how='left')
branch_summary['ghost_count'] = branch_summary['ghost_count'].fillna(0).astype(int)
branch_summary['ghost_rate'] = branch_summary['ghost_count'] / branch_summary['total_outlets'] * 100

# Sort by ghost rate
branch_summary = branch_summary.sort_values('ghost_rate', ascending=False)

print("\nTop 10 Branch dengan Ghost Rate Tertinggi:")
print(branch_summary.head(10).to_string(index=False))

In [ ]:
# ── EXPORT KE EXCEL (MULTIPLE SHEETS) ───────────────────────────────────
print("\n" + "="*60)
print("💾 EXPORTING TO EXCEL")
print("="*60)

# Buat timestamp untuk filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
excel_file = OUT_DIR / f"LURGIP_Report_{timestamp}.xlsx"

with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    # Sheet 1: Executive Summary
    summary_data = {
        'Metric': [
            'Total Outlet',
            f'Ghost Outlets (>{GHOST_DAYS} hari)',
            'Ghost Rate',
            f'Churn Risk (<{CHURN_THRESHOLD*100:.0f}%)',
            'Churn Rate',
            'Outlet Dikunjungi',
            'Rata-rata Kunjungan',
            'High Potential Prospects',
            'Medium Potential Prospects',
            'Low Potential Prospects'
        ],
        'Value': [
            total_outlets,
            ghost_count,
            f"{ghost_rate:.2f}%",
            churn_count,
            f"{churn_rate:.2f}%",
            outlets_with_visits,
            f"{avg_visits:.2f}",
            high_potential,
            medium_potential,
            low_potential
        ]
    }
    pd.DataFrame(summary_data).to_excel(writer, sheet_name='Executive Summary', index=False)
    
    # Sheet 2: Ghost Outlets
    df_ghost.to_excel(writer, sheet_name='Ghost Outlets', index=False)
    
    # Sheet 3: Route Compliance
    df_route.to_excel(writer, sheet_name='Route Compliance', index=False)
    
    # Sheet 4: Sales Performance
    df_sales.to_excel(writer, sheet_name='Sales Performance', index=False)
    
    # Sheet 5: Prospect Score
    df_prospect.to_excel(writer, sheet_name='Prospect Score', index=False)
    
    # Sheet 6: Branch Summary
    branch_summary.to_excel(writer, sheet_name='Branch Summary', index=False)

print(f"\n✅ Excel report saved: {excel_file.name}")
print(f"   Location: {excel_file}")
print(f"   Sheets: Executive Summary, Ghost Outlets, Route Compliance,")
print(f"           Sales Performance, Prospect Score, Branch Summary")

In [ ]:
# ── EXPORT TOP ISSUES (ACTIONABLE) ──────────────────────────────────────
print("\n" + "="*60)
print("⚠️  TOP ISSUES FOR ACTION")
print("="*60)

# Top 20 ghost outlets by branch
top_ghost = df_ghost.groupby(['szBranchId', 'szCustomerId']).size().reset_index(name='days_inactive')
top_ghost = top_ghost.sort_values('days_inactive', ascending=False).head(20)

print("\nTop 20 Ghost Outlets (perlu follow-up):")
print(top_ghost.to_string(index=False))

# Export ke CSV terpisah
top_ghost.to_csv(OUT_DIR / "top_ghost_outlets_action.csv", index=False)
print(f"\n✅ Saved: top_ghost_outlets_action.csv")

# Top 20 churn risk outlets
top_churn = df_sales[df_sales['churn_risk'] == True].sort_values('sales_change_pct').head(20)
top_churn_export = top_churn[['szCustomerId', 'current_sales', 'previous_sales', 'sales_change_pct']]
top_churn_export.to_csv(OUT_DIR / "top_churn_risk_action.csv", index=False)
print(f"✅ Saved: top_churn_risk_action.csv")

In [ ]:
# ── GENERATE TEXT REPORT ────────────────────────────────────────────────
print("\n" + "="*60)
print("📝 GENERATING TEXT REPORT")
print("="*60)

report_text = f"""
LURGIP MVP - LAPORAN ANALITIK DISTRIBUSI FMCG
================================================
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

RINGKASAN EKSEKUTIF
-------------------
Total Outlet              : {total_outlets:,}
Ghost Outlet (>{GHOST_DAYS} hr)  : {ghost_count:,} ({ghost_rate:.1f}%)
Churn Risk                : {churn_count:,} ({churn_rate:.1f}%)
Outlet Dikunjungi         : {outlets_with_visits:,}
Rata-rata Kunjungan       : {avg_visits:.1f}x

PROSPECT SCORING
----------------
High Potential (≥70)      : {high_potential:,}
Medium Potential (40-69)  : {medium_potential:,}
Low Potential (<40)       : {low_potential:,}

TOP BRANCH BY GHOST RATE
------------------------
{branch_summary.head(10).to_string(index=False)}

REKOMENDASI AKSI
----------------
1. Follow-up {ghost_count:,} ghost outlet (> {GHOST_DAYS} hari tanpa transaksi)
2. Retensi {churn_count:,} outlet dengan churn risk
3. Prioritaskan {high_potential:,} prospect high potential
4. Review route compliance untuk outlet dengan kunjungan irregular

FILES GENERATED
---------------
- LURGIP_Report_*.xlsx (Full report dengan multiple sheets)
- top_ghost_outlets_action.csv (Action list ghost outlet)
- top_churn_risk_action.csv (Action list churn risk)
- map_*.html (Interactive maps)

================================================
LURGIP MVP - Local Urban Retail Geographic Intelligence Platform
"""

# Simpan text report
report_file = OUT_DIR / f"LURGIP_Summary_{timestamp}.txt"
with open(report_file, 'w', encoding='utf-8') as f:
    f.write(report_text)

print(f"\n✅ Text report saved: {report_file.name}")
print("\n" + report_text)

In [ ]:
# ── FINAL SUMMARY ───────────────────────────────────────────────────────
print("\n" + "="*70)
print("✅ EXPORT & REPORT COMPLETE")
print("="*70)
print(f"\n📁 Semua file tersimpan di: {OUT_DIR}")
print("\n📊 Files generated:")
print("   1. LURGIP_Report_YYYYMMDD_HHMMSS.xlsx (Full report)")
print("   2. LURGIP_Summary_YYYYMMDD_HHMMSS.txt (Text summary)")
print("   3. top_ghost_outlets_action.csv (Action list)")
print("   4. top_churn_risk_action.csv (Action list)")
print("   5. map_*.html (5 interactive maps)")
print("\n🎯 Next Steps:")
print("   - Review executive summary dengan management")
print("   - Distribusikan action list ke tim sales")
print("   - Jadwalkan follow-up untuk ghost & churn risk outlets")
print("   - Run notebook ini monthly untuk tracking progress")
print("\n🚀 LURGIP MVP Ready for Production!")